# Введение в MapReduce модель на Python


In [1332]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [1333]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row) #генерация пары (возраст, строка данных)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]): #вычисление среднего количества социальных контрактов для каждой возрастной группы
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [1334]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [1335]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [1336]:
#функция имитирует процесс загрузки данных в MapReduce модель
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [1337]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [1338]:
#функция принимает вложенную коллекцию и "разворачивает" её. 
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [1339]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [1340]:
#все элементы с одинаковым ключом группируются вместе
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [1341]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [1342]:
#среднее количество социальных контрактов для каждой возрастной группы
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [1343]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [1344]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [1345]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [1346]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 3.067778657157726),
 (1, 3.067778657157726),
 (2, 3.067778657157726),
 (3, 3.067778657157726),
 (4, 3.067778657157726)]

## Inverted index 

In [1347]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [1348]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [1349]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [1350]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('what', 10)]),
 (1, [('is', 18), ('it', 18)])]

## TeraSort

In [1351]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.004441374397178177),
   (None, 0.04293826046436944),
   (None, 0.07644102732670477),
   (None, 0.11697092619554406),
   (None, 0.1632773107428268),
   (None, 0.1660337798314213),
   (None, 0.24355726451381077),
   (None, 0.2844180489789434),
   (None, 0.2901323565321041),
   (None, 0.29277057415914054),
   (None, 0.29488028108933106),
   (None, 0.3708053810135784),
   (None, 0.40308838183787465),
   (None, 0.4349037383404346),
   (None, 0.48935249281368287)]),
 (1,
  [(None, 0.5182607711334679),
   (None, 0.5208386870452214),
   (None, 0.5753388641657243),
   (None, 0.6115103919425058),
   (None, 0.6613781075073888),
   (None, 0.6856527992167252),
   (None, 0.6916171432430752),
   (None, 0.7272017026277521),
   (None, 0.8443654656215461),
   (None, 0.8590579490588727),
   (None, 0.8727835837539624),
   (None, 0.8827904008233813),
   (None, 0.8964075449711834),
   (None, 0.9340381027119073),
   (None, 0.9875723152390045)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [1352]:
# RECORDREADER: генерация данных для MapReduce
def RECORDREADER():
    return [(u.id, u) for u in input_collection]

# MAPMAX: для каждого пользователя передаем количество социальных контактов
def MAPMAX(_, row: NamedTuple):
    yield ("", row) 

# REDUCE_MAX: находим максимальное количество социальных контактов
def REDUCE_MAX(_, rows: Iterator[NamedTuple]):
    max_contacts = 0
    user_max_contacts = None
    for row in rows:
        if row.social_contacts > max_contacts:  # Сравниваем по количеству социальных контактов
            max_contacts = row.social_contacts
            user_max_contacts = row
    yield user_max_contacts

output = MapReduce(RECORDREADER, MAPMAX, REDUCE_MAX)

output = list(output)
print(output)


[User(id=3, age=33, social_contacts=800, gender='female')]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [1353]:
def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAPAVG(_, row: NamedTuple):
    yield ("", row) 

def MAPREDUCE(_, rows: Iterator[NamedTuple]):
    total_contacts = 0
    count = 0
    for row in rows:
        total_contacts += row.social_contacts 
        count += 1  
    average_contacts = total_contacts / count if count > 0 else 0  
    yield (average_contacts)

# Вызываем MapReduce
output = MapReduce(RECORDREADER, MAPAVG, MAPREDUCE)

# Печатаем результат
output = list(output)
print(output)

[390.0]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [1354]:
# groupbykey: Группировка данных по ключу (по возрасту)
def groupbykey(iterable):
    iterable = sorted(iterable, key=lambda e: e[0])  # Сортируем по ключу (возрасту)
    group = {}
    for k2, v2 in iterable:
        group[k2] = group.get(k2, []) + [v2]  # Группируем по ключу
    return group.items()

output = MapReduce(RECORDREADER, MAPMAX, REDUCE_MAX)
output = list(output)
print(f"MapReduce MAX: {output}")

output = MapReduce(RECORDREADER, MAPAVG, MAPREDUCE)
output = list(output)
print(f"MapReduce AVG: {output}")

MapReduce MAX: [User(id=3, age=33, social_contacts=800, gender='female')]
MapReduce AVG: [390.0]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [1355]:
input_collection = [
    User(id=0, age="41", gender='male', social_contacts=200),
    User(id=0, age="25", gender='female', social_contacts=240),
    User(id=1, age="29", gender='male', social_contacts=120),
    User(id=1, age="33", gender='female', social_contacts=800),
    User(id=2, age="25", gender='male', social_contacts=109),
    User(id=2, age="17", gender='female', social_contacts=100),
    User(id=3, age="22", gender='female', social_contacts=237)
]
# Функция для распределенной группировки с использованием сортировки
def distribute_grouping(map_partitions, partitioner):
    global reducers
    partitions = [{} for _ in range(reducers)]
    for partition in map_partitions:
        for key, value in partition:
            partition_idx = partitioner(key)
            partitions[partition_idx][key] = partitions[partition_idx].get(key, []) + [value]
    return [(idx, sorted(partition.items(), key=lambda item: item[0])) for idx, partition in enumerate(partitions)]

# Функция для вычисления партиции по хешу
def custom_partitioner(key):
    global reducers
    return hash(key) % reducers

# Основная функция MapReduce для распределенной обработки
def run_map_reduce(input_function, mapper, reducer, partitioner=custom_partitioner, combiner=None):
    mapped_partitions = map(lambda record_reader: flatten(map(lambda k1v1: mapper(*k1v1), record_reader)), input_function())
    
    if combiner:
        mapped_partitions = map(lambda partition: flatten(map(lambda k2v2: combiner(*k2v2), group_by_key(partition))), mapped_partitions)
    
    reduce_partitions = distribute_grouping(mapped_partitions, partitioner)
    
    reduced_output = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input: reducer(*reduce_input), reduce_partition[1]))), reduce_partitions)
    
    print(f"{sum([len(values) for _, values in flatten([partition for _, partition in reduce_partitions])])} key-value pairs were sent over the network.")
    return reduced_output

# Устанавливаем параметры для MapReduce
maps = 3
reducers = 2

# Чтение и разделение данных
def input_function():
    global maps

    def record_reader(split):
        for user_id, user in enumerate(split):
            yield (user.id, user)

    split_size = int(np.ceil(len(input_collection) / maps))
    for i in range(0, len(input_collection), split_size):
        yield record_reader(input_collection[i: i + split_size])

# MAP для уникальности: передаем объект с уникальным идентификатором
def map_unique(_, user: User):
    yield (user.id, user)

# REDUCE для уникальности: устраняем дубликаты, оставляем только уникальные объекты
def reduce_unique(user_id: int, users: Iterator[User]):
    unique_persons = {user for user in users}  # Используем set для исключения дубликатов
    yield list(unique_persons)[0]  # Возвращаем одного уникального пользователя

# Запуск MapReduce для исключения дублей
partitioned_output = run_map_reduce(input_function, map_unique, reduce_unique, combiner=None)
partitioned_output = [(partition_id, list(partition)) for partition_id, partition in partitioned_output]
print(partitioned_output)

7 key-value pairs were sent over the network.
[(0, [User(id=0, age='25', social_contacts=240, gender='female'), User(id=2, age='25', social_contacts=109, gender='male')]), (1, [User(id=1, age='29', social_contacts=120, gender='male'), User(id=3, age='22', social_contacts=237, gender='female')])]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [1356]:
def RECORDREADER():
    return [(u.id, u) for u in input_collection]
def MAPSELECT(_, row: NamedTuple):
    if row.gender == "female":
        yield (row, row)
def REDUCESELECT(row: str, rows: Iterator[NamedTuple]):
    yield (row.gender, rows)

output = MapReduce(RECORDREADER, MAPSELECT, REDUCESELECT)
output = list(output)
output

[('female', [User(id=0, age='25', social_contacts=240, gender='female')]),
 ('female', [User(id=1, age='33', social_contacts=800, gender='female')]),
 ('female', [User(id=2, age='17', social_contacts=100, gender='female')]),
 ('female', [User(id=3, age='22', social_contacts=237, gender='female')])]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [1357]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=2, age=35, gender='male', social_contacts=490),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAPPROJECTION(_, row: NamedTuple):
    yield (row.id, {'id': row.id, 'age': int(row.age), 'gender': row.gender, 'social_contacts': row.social_contacts})

def REDUCEPROJECTION(id: int, users: Iterator[dict]):
    user_list = list(users)  # собираем все данные для одного id в список
    yield (id, user_list)

output = MapReduce(RECORDREADER, MAPPROJECTION, REDUCEPROJECTION)
output = list(output)
output



[(0, [{'id': 0, 'age': 55, 'gender': 'male', 'social_contacts': 20}]),
 (1, [{'id': 1, 'age': 25, 'gender': 'female', 'social_contacts': 240}]),
 (2,
  [{'id': 2, 'age': 25, 'gender': 'female', 'social_contacts': 500},
   {'id': 2, 'age': 35, 'gender': 'male', 'social_contacts': 490}]),
 (3, [{'id': 3, 'age': 33, 'gender': 'female', 'social_contacts': 800}])]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [1358]:
input_collection_1 = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=2, age=35, gender='male', social_contacts=490),
    User(id=3, age=33, gender='female', social_contacts=800)
]
input_collection_2 = [
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=41, gender='female', social_contacts=230)
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection_1 + input_collection_2]

def MAP_UNION(_, row: NamedTuple):
    yield (row.id, row)

def REDUCE_UNION(row: str, rows: Iterator[NamedTuple]):
    yield (rows[0], rows[0])

output = MapReduce(RECORDREADER, MAP_UNION, REDUCE_UNION)
output = list(output)
output

[(User(id=0, age=55, social_contacts=20, gender='male'),
  User(id=0, age=55, social_contacts=20, gender='male')),
 (User(id=1, age=25, social_contacts=240, gender='female'),
  User(id=1, age=25, social_contacts=240, gender='female')),
 (User(id=2, age=25, social_contacts=500, gender='female'),
  User(id=2, age=25, social_contacts=500, gender='female')),
 (User(id=3, age=33, social_contacts=800, gender='female'),
  User(id=3, age=33, social_contacts=800, gender='female')),
 (User(id=4, age=41, social_contacts=230, gender='female'),
  User(id=4, age=41, social_contacts=230, gender='female'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [1359]:
def RECORDREADER():
    return [(u.id, u) for u in input_collection_1 + input_collection_2]

def MAPINTERSECTION(_, row: NamedTuple):
    yield (row.id, row)

def REDUCEINTERSECTION(row_id: int, rows: Iterator[NamedTuple]):
    if len(rows) == 2:
        yield rows

output = MapReduce(RECORDREADER, MAPINTERSECTION, REDUCEINTERSECTION)
output = list(output)
output

[[User(id=2, age=25, social_contacts=500, gender='female'),
  User(id=2, age=35, social_contacts=490, gender='male')],
 [User(id=3, age=33, social_contacts=800, gender='female'),
  User(id=3, age=33, social_contacts=800, gender='female')]]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [1360]:
def RECORDREADER():
    return [(0, a) for a in input_collection_1] + [(1, b) for b in input_collection_2]

def MAP_DIFFERENCE(collection_id, user):
    yield (user, collection_id)

def REDUCE_DIFFERENCE(user, collections):
    if collections == [0]:
        yield (user)


output = MapReduce(RECORDREADER, MAP_DIFFERENCE, REDUCE_DIFFERENCE)
output = list(output)
output

[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=1, age=25, social_contacts=240, gender='female'),
 User(id=2, age=25, social_contacts=500, gender='female'),
 User(id=2, age=35, social_contacts=490, gender='male')]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [1361]:
class User(NamedTuple):
    id: int
    age: str
    gender: str
    social_contacts: int
    job_id: int

class JobTitle(NamedTuple):
    id: int
    name: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20, job_id=0),
    User(id=1, age=25, gender='female', social_contacts=240, job_id=2),
    User(id=2, age=25, gender='female', social_contacts=500, job_id=2),
    User(id=2, age=35, gender='male', social_contacts=490, job_id=1),
    User(id=3, age=33, gender='female', social_contacts=800, job_id=0)
]

job_collection = [
    JobTitle(id=0, name="Doctor"),
    JobTitle(id=1, name="Programmer"),
    JobTitle(id=2, name="Blogger"),
]

def RECORDREADER():
    return [(user.job_id, user) for user in input_collection] + [(job.id, job) for job in job_collection]

def MAPJOIN(job_id, row):
    yield (job_id, row)

def REDUCEJOIN(job_id, rows):
    users = []
    job = None

    for row in rows:
        if type(row) is User:
            users += [row]
        else:
            job = row

    for row in rows:
        if type(row) is User:
            yield (row, row.job_id, job)

output = MapReduce(RECORDREADER, MAPJOIN, REDUCEJOIN)
output = list(output)
join = output
join

[(User(id=0, age=55, gender='male', social_contacts=20, job_id=0),
  0,
  JobTitle(id=0, name='Doctor')),
 (User(id=3, age=33, gender='female', social_contacts=800, job_id=0),
  0,
  JobTitle(id=0, name='Doctor')),
 (User(id=2, age=35, gender='male', social_contacts=490, job_id=1),
  1,
  JobTitle(id=1, name='Programmer')),
 (User(id=1, age=25, gender='female', social_contacts=240, job_id=2),
  2,
  JobTitle(id=2, name='Blogger')),
 (User(id=2, age=25, gender='female', social_contacts=500, job_id=2),
  2,
  JobTitle(id=2, name='Blogger'))]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [1362]:
def RECORDREADER():
    return [(job_id, user, job) for user, job_id, job in join]

def MAPGROUP(job_id, user, job):
    yield (job_id, user)

def REDUCEGROUP(job_id, rows):
    yield f"job id={job_id} = {len(rows)} user(s)"


output = MapReduce(RECORDREADER, MAPGROUP, REDUCEGROUP)
output = list(output)
output

['job id=0 = 2 user(s)', 'job id=1 = 1 user(s)', 'job id=2 = 2 user(s)']

# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [1363]:
def RECORDREADER():
    return [(None, m) for m in input_matrix]
input_matrix = [
    (1, 2, 3), (4, 5, 6), (7, 8, 9),
    (1, 2, 3), (4, 5, 6), (7, 8, 9),
    (1, 2, 3), (4, 5, 6), (7, 8, 9),
]

input_vector = [
    (1, 2), (3, 4), (5, 6)
]

def MAPMATRIXVECTOR(_, matrix_row):
    row, col, value = matrix_row
    for vector_col, vector_value in input_vector:
        if vector_col == col:
            yield (row, value * vector_value)


def REDUCEMATRIXVECTOR(row, values: Iterator[int]):
    yield (row, sum(values))


output = MapReduce(RECORDREADER, MAPMATRIXVECTOR, REDUCEMATRIXVECTOR)
output = list(output)
output

[(4, 108)]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [1364]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [1365]:
import numpy as np

I = 2  
J = 3  
K = 4 * 10  

small_mat = np.random.rand(I, J)  
big_mat = np.random.rand(J, K)   

# Функция RECORDREADER для создания пар (j, k) с элементами большой матрицы
def RECORDREADER():
    for j in range(big_mat.shape[0]):  # Перебор строк большой матрицы
        for k in range(big_mat.shape[1]):  # Перебор столбцов
            yield ((j, k), big_mat[j, k])  # Генерация пары (j, k) с элементом большой матрицы

# MAP-функция для создания пар (i, k) с элементами маленькой и большой матрицы
def MAP(k1, v1):
    (j, k) = k1  
    w = v1       
    # Для каждой строки малой матрицы генерируем пару (i, k) с элементами обеих матриц
    for i in range(I):  # Перебор строк маленькой матрицы
        # Создаем пару (i, k) с соответствующими элементами обеих матриц
        yield ((i, k), (small_mat[i, j], w))

# REDUCE-функция для агрегации результатов умножения
def REDUCE(key, values):
    (i, k) = key  # Получаем индексы (i, k) для итоговой матрицы
    result = 0
    for (small_value, big_value) in values:
        result += small_value * big_value  # Суммируем произведения для каждого (i, k)
    yield ((i, k), result)  # Возвращаем результат умножения



Проверьте своё решение

In [1366]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [1367]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [1368]:
I = 2  
J = 3  
K = 4 * 10  

small_mat = np.random.rand(I, J)  
big_mat = np.random.rand(J, K)   

# Генерация элементов обеих матриц для MapReduce
def RECORDREADER():
    for row in range(I):
        for col in range(J):
            for idx in range(K):
                yield (((row, col), small_mat[row, col]), ((col, idx), big_mat[col, idx]))

# Перемножение элементов матриц с помощью Map
def MAP(item1, item2):
    (i, j), value_small = item1
    (j, k), value_big = item2
    # Умножение значений матриц и передача результата
    yield ((i, k), value_small * value_big)

def REDUCE(key, values):
    (i,k) = key
    total_sum = sum(values)  # Суммируем результаты умножения
    yield (key, total_sum)

solution = MapReduce(RECORDREADER, MAP, REDUCE)

reference_solution = np.matmul(small_mat, big_mat) 
np.allclose(reference_solution, asmatrix(solution)) # should return true

True

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [1369]:
I = 2  
J = 3  
K = 4 * 10 

small_mat = np.random.rand(I, J)
big_mat = np.random.rand(J, K)

def INPUTFORMAT():
    def RECORDREADER(i_range):
        for i in i_range:
            for j in range(J):
                for k in range(K):
                    yield (((i, j), small_mat[i, j]), ((j, k), big_mat[j, k]))

    # Разбиваем данные на части для каждой карты
    split_size = int(np.ceil(I / maps))
    for i in range(0, I, split_size):
        yield RECORDREADER(range(i, min(i + split_size, I)))

def MAP(element1, element2):
    (i, j), v1 = element1
    (j, k), v2 = element2

    # Вычисление произведения элементов двух матриц
    yield ((i, k), v1 * v2)

def REDUCE(key, values):
    (i, k) = key
    v3 = sum(values)  # Суммируем все произведения для каждой пары (i, k)
    yield ((i, k), v3)


def partitioner(key):
    return hash(key) % reducers  # Разделяем данные на несколько частей с помощью хеширования

reference_solution = np.matmul(small_mat, big_mat)

# Используем MapReduceDistributed для выполнения распределенной обработки
maps = 2
reducers = 2
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for partition_id, partition in partitioned_output]

# Собираем все данные из разных частей
solution = []
for output_part in partitioned_output:
    for element in output_part[1]:
        solution.append(element)

# Проверка корректности
print(np.allclose(reference_solution, asmatrix(solution))) 
print(asmatrix(solution))


240 key-value pairs were sent over a network.
True
[[1.21196237 1.80627987 1.45570664 0.63156185 1.19424242 0.88753051
  1.1846396  1.43219504 1.62589935 1.40656309 1.8858621  1.24027472
  0.5728774  1.35990022 1.10002994 1.20896893 0.39256304 0.94710429
  1.42042377 0.85014158 1.01284097 1.14610398 0.83876027 0.67470101
  1.73796164 1.41367745 1.20452595 0.44851628 1.74212924 1.57567147
  1.59261382 0.97588454 1.17946735 1.62689858 1.17722777 1.50752102
  1.13931808 1.594247   0.82885465 1.43199896]
 [1.06391955 1.66975119 1.652785   0.72903359 1.24219063 1.12554807
  1.12035004 1.47251312 1.38969726 1.09310244 1.45497763 0.94629454
  0.28315311 1.37397956 1.22440953 0.58374698 0.30215015 0.57482669
  1.45072039 1.08324309 0.35292337 1.04326038 1.04681505 0.39800097
  1.18897996 1.28261681 1.09885215 0.4096398  1.40641602 1.30236993
  1.46443974 0.71562152 1.01124027 1.00652683 1.3224448  0.92335878
  1.3434956  0.88351857 0.96804726 1.19827473]]


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [1370]:
# Да, можно обобщить предыдущие решения так, чтобы несколько RECORDREADER-ов генерировали подмножества элементов каждой матрицы. 
# Для этого можно расширить решение, чтобы работать с несколькими генераторами данных для каждой матрицы, каждый из которых будет генерировать случайное подмножество элементов.
# Однако важно понимать, что результат может зависеть от того, как именно распределяются элементы матрицы. 
# В случае, когда каждый RECORDREADER генерирует только подмножество элементов матрицы, необходимо проверить, что все возможные индексы матрицы будут правильно обработаны. 